#Referencing

###Part A
One of the targets of this project is to map the tranportational hubs where investments are required. To do that, we need to know the existing transportational hubs. 

###Part B
Other target is to analyze daily milage per vehicle and trip distributions to simulate which routes are ready for EVs based on battery range limits

##Working on Part A

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

trips_final_staged = (
    spark.table("workspace.silver.trips_clean")
    .withColumn("trip_id", F.monotonically_increasing_id())
)

trips_final_staged.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.silver.trips_with_id")

trips_final = spark.table("workspace.silver.trips_with_id")

Dataset related to subways were included into the data volume.

**Assumption**: Only subway stations were considered as major transportational hubs

In [0]:
hub_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/bronze/raw_files/reference/MTA_Subway_Stations_and_Complexes_20260918.csv")
)

hub_raw.printSchema()

root
 |-- Complex ID: integer (nullable = true)
 |-- Is Complex: boolean (nullable = true)
 |-- Number Of Stations In Complex: integer (nullable = true)
 |-- Stop Name: string (nullable = true)
 |-- Display Name: string (nullable = true)
 |-- Constituent Station Names: string (nullable = true)
 |-- Station IDs: string (nullable = true)
 |-- GTFS Stop IDs: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- CBD: boolean (nullable = true)
 |-- Daytime Routes: string (nullable = true)
 |-- Structure Type: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- ADA: integer (nullable = true)
 |-- ADA Notes: string (nullable = true)



In [0]:
hub_raw.show(5, truncate=False)

+----------+----------+-----------------------------+--------------------+------------------------+-------------------------+-----------+-------------+-------+-----+--------------+--------------+---------+----------+---+-------------+
|Complex ID|Is Complex|Number Of Stations In Complex|Stop Name           |Display Name            |Constituent Station Names|Station IDs|GTFS Stop IDs|Borough|CBD  |Daytime Routes|Structure Type|Latitude |Longitude |ADA|ADA Notes    |
+----------+----------+-----------------------------+--------------------+------------------------+-------------------------+-----------+-------------+-------+-----+--------------+--------------+---------+----------+---+-------------+
|398       |false     |1                            |77 St               |77 St (6)               |77 St                    |398        |627          |M      |false|6             |Subway        |40.77362 |-73.959874|0  |NULL         |
|399       |false     |1                            |68 St-H

In [0]:
H3_RESOLUTION = 8

In [0]:
dim_transit_hub = (
    hub_raw.select(
        F.col("Complex ID").alias("hub_id"),
        F.col("Stop Name").alias("hub_name"),
        F.col("Borough").alias("hub_borough"),
        F.col("Latitude").cast("double").alias("hub_lat"),
        F.col("Longitude").cast("double").alias("hub_lon")
    )
    .dropDuplicates(["hub_id"])
    .filter(F.col("hub_lat").isNotNull() & F.col("hub_lon").isNotNull())
    .withColumn(
        "hub_h3", F.expr(f"h3_longlatash3string(hub_lon, hub_lat, {H3_RESOLUTION})")
    )
)

dim_transit_hub.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.silver.dim_transit_hub")

Based on the pickup_h3s and dropoff_h3s, k=1 ring is generated and hubs will be matched inside that ring

###Creating an algorithm to detect the nearest hub

* K-ring/H3 component filters "approximate candidates" by identifying nearby hubs. 
* **_Harversine formula_** calculates the exact distance (in km) between these candidates. By combining these two, a "fast and accurate" nearest neighbour is created

**_Harversine Formula_**
![image_1789706778528.png](./image_1789706778528.png "image_1789706778528.png")

In [0]:
EARTH_RADIUS_KM = 6371.0

In [0]:
# Creating a function to calculate the nearest hub
def get_nearest_hub(trips_df, h3_col, lat_col, lon_col, prefix):
    candidates = (
        trips_df
        .select("trip_id", h3_col, lat_col, lon_col)
        .withColumn("kring_cells", F.expr(f"h3_kring({h3_col}, 1)"))
        .withColumn("candidate_h3", F.explode("kring_cells"))
        .join(
            F.broadcast(dim_transit_hub.select("hub_id", "hub_h3", "hub_lat", "hub_lon")),
            F.col("candidate_h3") == F.col("hub_h3"),
            "inner"
        )
    )

    # Haversine Formula
    R = EARTH_RADIUS_KM

    b = F.pow(F.sin(F.radians(F.col("hub_lat") - F.col(lat_col)) / 2), 2)
    c = F.pow(F.sin(F.radians(F.col("hub_lon") - F.col(lon_col)) / 2), 2) \
        * F.cos(F.radians(F.col(lat_col))) * F.cos(F.radians(F.col("hub_lat")))
    a = b + c

    dist_expr = F.lit(2) * R * F.asin(F.sqrt(a))

    candidates = candidates.withColumn("distance_km", dist_expr)

    nearest = (
        candidates
        .groupBy("trip_id")
        .agg(F.min("distance_km").alias(f"{prefix}_nearest_hub_km"))
    )

    return nearest

In [0]:
pickup_hub = get_nearest_hub(trips_final, "pickup_h3", "pickup_lat", "pickup_lon", "pickup")
dropoff_hub = get_nearest_hub(trips_final, "dropoff_h3", "dropoff_lat", "dropoff_lon", "dropoff")

trips_with_hub = (
    trips_final
    .join(pickup_hub, "trip_id", "left")
    .join(dropoff_hub, "trip_id", "left")
    .withColumn(
        "nearest_hub_km",
        F.least(F.col("pickup_nearest_hub_km"), F.col("dropoff_nearest_hub_km"))
    )
    .withColumn(
        "is_hub_connected",
        F.col("pickup_nearest_hub_km").isNotNull() | F.col("dropoff_nearest_hub_km").isNotNull()
    )
)

In [0]:
trips_final.show()

+-------------+-------------------+-------------------+-----------+--------------+-------------+-------------+-----------+----------------------+---------+--------------+-----------+------------------+----------+-----------------------+------------+-------------------+-----------+---------------------+--------------------+------------+---------+------------------+------------------+--------------+---------+------------------+------------------+---------------+---------------+---------------+------------------+------------------+-------------+-------------+-------------------+-----------------+-----------+
|provider_code|   pickup_timestamp|  dropoff_timestamp|rider_count|distance_miles|rate_class_id|origin_loc_id|dest_loc_id|fare_settlement_method|base_fare|surcharge_misc|transit_tax|driver_tip_payment|toll_total|service_improvement_fee|charge_total|zone_congestion_fee|Airport_fee|congestion_relief_fee|         source_file|source_month|pu_loc_id|        pickup_lat|        pickup_lon|pi

In [0]:
trips_final.orderBy("trip_id").show()

+-------------+-------------------+-------------------+-----------+--------------+-------------+-------------+-----------+----------------------+---------+--------------+-----------+------------------+----------+-----------------------+------------+-------------------+-----------+---------------------+--------------------+------------+---------+------------------+------------------+--------------+---------+------------------+------------------+---------------+---------------+---------------+------------------+------------------+-------------+-------------+-------------------+-----------------+-------+
|provider_code|   pickup_timestamp|  dropoff_timestamp|rider_count|distance_miles|rate_class_id|origin_loc_id|dest_loc_id|fare_settlement_method|base_fare|surcharge_misc|transit_tax|driver_tip_payment|toll_total|service_improvement_fee|charge_total|zone_congestion_fee|Airport_fee|congestion_relief_fee|         source_file|source_month|pu_loc_id|        pickup_lat|        pickup_lon|pickup

In [0]:
print("Total trips:", trips_final.count())
print("Trips with hub match:", trips_with_hub.filter(F.col("is_hub_connected")).count())
print("Distinct trip_id in trips_final:", trips_final.select("trip_id").distinct().count())
print("Distinct trip_id in trips_with_hub:", trips_with_hub.select("trip_id").distinct().count())

Total trips: 48601782
Trips with hub match: 47814182
Distinct trip_id in trips_final: 48601782
Distinct trip_id in trips_with_hub: 48601782


Inspecting the Non connected trips

In [0]:
trips_with_hub.filter(~F.col("is_hub_connected")) \
 .groupBy("pickup_borough") \
 .count() \
 .orderBy(F.desc("count")) \
 .show()

+--------------+------+
|pickup_borough| count|
+--------------+------+
|        Queens|697434|
|       Unknown| 33573|
|           N/A| 20838|
|      Brooklyn| 17819|
|           EWR|  6262|
|     Manhattan|  5843|
|         Bronx|  5430|
| Staten Island|   401|
+--------------+------+



In [0]:
trips_final_staged.filter(F.col("pickup_borough") == "Staten Island").count()

5156

In [0]:
total_by_borough = trips_final_staged.groupBy("pickup_borough").count().withColumnRenamed("count", "total_trips")

not_connected_by_borough = (
    trips_with_hub
    .filter(~F.col("is_hub_connected"))
    .groupBy("pickup_borough")
    .count()
    .withColumnRenamed("count", "not_connected_trips")
)

connectivity_summary = (
    total_by_borough
    .join(not_connected_by_borough, "pickup_borough", "left")
    .fillna(0, subset=["not_connected_trips"])
    .withColumn(
        "not_connected_rate_pct",
        F.round(F.col("not_connected_trips") / F.col("total_trips") * 100, 2)
    )
    .orderBy(F.desc("not_connected_rate_pct"))
)
connectivity_summary.show()

+--------------+-----------+-------------------+----------------------+
|pickup_borough|total_trips|not_connected_trips|not_connected_rate_pct|
+--------------+-----------+-------------------+----------------------+
|           EWR|       6420|               6262|                 97.54|
|           N/A|      22560|              20838|                 92.37|
|       Unknown|      70293|              33573|                 47.76|
|        Queens|    4676678|             697434|                 14.91|
| Staten Island|       5156|                401|                  7.78|
|         Bronx|     419151|               5430|                   1.3|
|      Brooklyn|    1926540|              17819|                  0.92|
|     Manhattan|   41474984|               5843|                  0.01|
+--------------+-----------+-------------------+----------------------+



* EWR is an Airport. There are no subway stations near airports. So, 97% is acceptable
* N/A, Unknown boroughs mostly match with the records that wasn't matched in the zone lookup. They were flagged in the silver layer

#Gold Layer implementation decision: 
* Manhattan, Brooklyn, Bronx, Queens, Staten Island will be taken to create the hotspot.
* EWR will not be included as it's an airport area and it's located outside NYC area

##Working on Part B

Shift based chaining: Calculating sequential trips using a selected time gap and spatial proximity constrains

In [0]:
from pyspark.sql.window import Window

shift_window = (
    Window.partitionBy("provider_code", F.to_date("pickup_timestamp"))
          .orderBy("pickup_timestamp", "trip_id")
)

trips_shift = (
    trips_with_hub
    .withColumn("prev_dropoff_time", F.lag("dropoff_timestamp").over(shift_window))
    .withColumn("prev_dropoff_lat", F.lag("dropoff_lat").over(shift_window))
    .withColumn("prev_dropoff_lon", F.lag("dropoff_lon").over(shift_window))
    .withColumn(
        "time_gap_min",
        (F.unix_timestamp("pickup_timestamp") - F.unix_timestamp("prev_dropoff_time")) / 60
    )
)

In [0]:
ASSUMED_SPEED_KMH = 20.0

In [0]:
R = EARTH_RADIUS_KM

b = F.pow(F.sin(F.radians(F.col("pickup_lat") - F.col("prev_dropoff_lat")) / 2), 2)
c = F.pow(F.sin(F.radians(F.col("pickup_lon") - F.col("prev_dropoff_lon")) / 2), 2) \
    * F.cos(F.radians(F.col("prev_dropoff_lat"))) * F.cos(F.radians(F.col("pickup_lat")))
a = b + c

trips_shift = trips_shift.withColumn(
    "reposition_distance_km", F.lit(2) * R * F.asin(F.sqrt(a))
)

trips_shift = trips_shift.withColumn(
    "max_plausible_distance_km",
    ASSUMED_SPEED_KMH * (F.col("time_gap_min") / 60.0)
)

trips_shift = trips_shift.withColumn(
    "spatial_match",
    F.when(
        F.col("prev_dropoff_lat").isNotNull(),
        F.col("reposition_distance_km") <= F.col("max_plausible_distance_km")
    ).otherwise(F.lit(False))
)

In [0]:
trips_shift = trips_shift.withColumn(
    "is_new_shift",
    F.when(F.col("prev_dropoff_time").isNull(), 1)   # partition එකේ first trip
     .when(
        (F.col("time_gap_min") > 30) | (F.col("time_gap_min") < 0) | (~F.col("spatial_match")),
        1
     )
     .otherwise(0)
)

In [0]:
trips_shift = trips_shift.withColumn(
    "shift_id",
    F.sum("is_new_shift").over(
        shift_window.rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
)

In [0]:
trips_enriched = trips_shift.withColumn(
    "vehicle_id",
    F.concat_ws("_",
        F.col("provider_code").cast("string"),
        F.date_format("pickup_timestamp", "yyyyMMdd"),
        F.col("shift_id").cast("string")
    )
)

In [0]:
#Check up 1
trips_per_vehicle = (
    trips_enriched.groupBy("vehicle_id")
    .agg(F.count("trip_id").alias("trip_count"))
)

trips_per_vehicle.select(
    F.min("trip_count").alias("min"),
    F.round(F.avg("trip_count"), 2).alias("avg"),
    F.max("trip_count").alias("max"),
    F.expr("percentile_approx(trip_count, 0.5)").alias("median")
).show()

+---+---+---+------+
|min|avg|max|median|
+---+---+---+------+
|  1|1.0|  8|     1|
+---+---+---+------+



In [0]:
#Check up 2
daily_mileage = (
    trips_enriched.groupBy("vehicle_id")
    .agg(F.sum("distance_miles").alias("total_daily_distance"))
)

daily_mileage.select(
    F.round(F.avg("total_daily_distance"), 2).alias("avg_daily_miles"),
    F.round(F.expr("percentile_approx(total_daily_distance, 0.95)"), 2).alias("p95_daily_miles")
).show()

+---------------+---------------+
|avg_daily_miles|p95_daily_miles|
+---------------+---------------+
|           6.86|          12.93|
+---------------+---------------+



In [0]:
trips_final_staged.groupBy("provider_code", F.to_date("pickup_timestamp").alias("pickup_date")) \
    .count() \
    .agg(
        F.round(F.avg("count"), 0).alias("avg_trips_per_provider_per_day"),
        F.max("count").alias("max_trips_per_provider_per_day")
    ).show()

+------------------------------+------------------------------+
|avg_trips_per_provider_per_day|max_trips_per_provider_per_day|
+------------------------------+------------------------------+
|                       33198.0|                        146923|
+------------------------------+------------------------------+



avg_trips_per_provider_per_day = 33198 means ther

**Diagnosed problems**: There are thousands of records for one (provider_code + date) key per day (technically = 33198). Which means there are more than 1 vehicles do transportaion at the same time under the same provider. So, calculating time wise trips is extremely difficult.  

**Decision**: Simple Random Assignment.

In [0]:
scale_check = (
    trips_final_staged.groupBy("provider_code", F.to_date("pickup_timestamp").alias("pickup_date"))
    .count()
    .agg(
        F.round(F.avg("count"), 0).alias("avg_trips_per_provider_per_day"),
        F.max("count").alias("max_trips_per_provider_per_day")
    )
)
scale_check.show()

+------------------------------+------------------------------+
|avg_trips_per_provider_per_day|max_trips_per_provider_per_day|
+------------------------------+------------------------------+
|                       33198.0|                        146923|
+------------------------------+------------------------------+



In [0]:
AVG_TRIPS_PER_VEHICLE_PER_DAY = 27 

In [0]:
vehicle_counts = (
    trips_with_hub.filter("is_valid_trip").groupBy("provider_code", F.to_date("pickup_timestamp").alias("pickup_date"))
    .count()
    .withColumn(
        "vehicle_count",
        F.greatest(F.ceil(F.col("count") / AVG_TRIPS_PER_VEHICLE_PER_DAY), F.lit(1))   # min 1 vehicle
    )
    .select("provider_code", "pickup_date", "vehicle_count")
)

In [0]:
trips_random_sim = (
    trips_with_hub
    .withColumn("pickup_date", F.to_date("pickup_timestamp"))
    .join(vehicle_counts, ["provider_code", "pickup_date"], "left")
    .withColumn(
        "vehicle_index",
        F.floor(F.rand(seed=42) * F.col("vehicle_count"))
    )
)

In [0]:
trips_enriched = trips_random_sim.withColumn(
    "vehicle_id",
    F.concat_ws("_",
        F.col("provider_code").cast("string"),
        F.date_format(F.col("pickup_date"), "yyyyMMdd"),
        F.col("vehicle_index").cast("string")
    )
)

In [0]:
#Check up 1
trips_per_vehicle = (
    trips_enriched.filter("is_valid_trip").groupBy("vehicle_id")
    .agg(F.count("trip_id").alias("trip_count"))
)

trips_per_vehicle.select(
    F.min("trip_count").alias("min"),
    F.round(F.avg("trip_count"), 2).alias("avg"),
    F.max("trip_count").alias("max"),
    F.expr("percentile_approx(trip_count, 0.5)").alias("median")
).show()

+---+-----+---+------+
|min|  avg|max|median|
+---+-----+---+------+
|  1|26.99| 56|    27|
+---+-----+---+------+



In [0]:
#Check up 2
daily_mileage_valid = (
    trips_enriched
    .filter(F.col("is_valid_trip"))
    .groupBy("vehicle_id")
    .agg(F.sum("distance_miles").alias("total_daily_distance"))
)

daily_mileage_valid.select(
    F.round(F.avg("total_daily_distance"), 2).alias("avg_daily_miles"),
    F.round(F.expr("percentile_approx(total_daily_distance, 0.95)"), 2).alias("p95_daily_miles")
).show()

+---------------+---------------+
|avg_daily_miles|p95_daily_miles|
+---------------+---------------+
|          95.46|         148.56|
+---------------+---------------+



In [0]:
trips_enriched.filter(~F.col("is_valid_trip")) \
    .agg(
        F.count("*").alias("invalid_trip_count"),
        F.max("distance_miles").alias("max_distance"),
        F.round(F.avg("distance_miles"), 2).alias("avg_distance_invalid")
    ).show()

trips_enriched.filter(~F.col("is_valid_trip")) \
    .groupBy("vehicle_id") \
    .agg(F.sum("distance_miles").alias("distance_from_invalid_trips")) \
    .orderBy(F.desc("distance_from_invalid_trips")) \
    .show(10)

+------------------+------------+--------------------+
|invalid_trip_count|max_distance|avg_distance_invalid|
+------------------+------------+--------------------+
|           2908613|   397994.37|               58.97|
+------------------+------------+--------------------+

+---------------+---------------------------+
|     vehicle_id|distance_from_invalid_trips|
+---------------+---------------------------+
|2_20250707_1282|                  397994.37|
|2_20250425_3268|                  386088.43|
|2_20250402_3118|                  335783.69|
| 2_20260220_394|         328522.22000000003|
|2_20251221_2413|                  322576.17|
|2_20251213_2604|                  321242.41|
|2_20251213_1234|                  321242.41|
|2_20251207_2052|                  320206.74|
|2_20250922_1949|                  318608.57|
|2_20250907_2529|                  317011.01|
+---------------+---------------------------+
only showing top 10 rows


In [0]:
trips_enriched.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.silver.trips_enriched")